In [53]:
from spark_utils import SparkUtils
su = SparkUtils()

In [54]:
# show the data


column_types = [("timestamp_received", "long"),
                ("timestamp_created_at", "long"),
                ("market_id", "string"),
                ("best_bid", "double"),
                ("best_ask", "float"),
                ("change_price", "float"),
                ("change_size", "float"),
                ("change_side", "string"),
                ("token_id", "string"),
                ("spread", "float"),
                ("mid_price", "float")
                ]


order_book = SparkUtils.generate_schema(column_types)
order_book_df = su._spark \
                .read \
                .schema(order_book) \
                .parquet("/opt/spark/work-dir/data/orderbooks/")


In [55]:
#how many entries are


In [56]:
from pyspark.sql import functions as F
column_types = [("timestamp_received", "long"),
                ("timestamp_created_at", "long"),
                ("market_id", "string"),
                ("update_type", "string"),
                ("data", "string")
                ]

json_data = [
                ("token_id","string"),
                ("side", "string"),
                ("best_bid", "string"),
                ("best_ask", "string"),
                ("timestamp", "float"),
                ("bids", "array_string"),
                ("asks", "array_string")
]


snapshots_schema = SparkUtils.generate_schema(column_types)
snapshots= su._spark.read.schema(snapshots_schema).parquet("/opt/spark/work-dir/data/snapshots/")

json_data = SparkUtils.generate_schema(json_data)

snapshots = (
    snapshots
    .withColumn("data_parsed", F.from_json(F.col("data"), json_data))
    .withColumn("token_id", F.col("data_parsed.token_id"))
    .withColumn("side", F.col("data_parsed.side"))
    .withColumn("best_bid", F.col("data_parsed.best_bid").cast("double"))
    .withColumn("best_ask", F.col("data_parsed.best_ask").cast("double"))
    .withColumn("book_timestamp", F.col("data_parsed.timestamp"))
    .withColumn("bids", F.col("data_parsed.bids"))
    .withColumn("asks", F.col("data_parsed.asks"))
    .drop("data", "data_parsed")
)
#show data column


In [57]:
column_types = [("condition_id", "string"),
                ("question", "string"),
                ("end_date", "string"),
                ("closed", "boolean"),
                ("uma_status", "string"),
                ("liquidity","double"),
                ("clob_token_id_yes", "string"),
                ("clob_token_id_no", "string")]

targets_schema = SparkUtils.generate_schema(column_types)

targets = (
    su._spark.read
    .schema(targets_schema)
    .parquet("/opt/spark/work-dir/data/labels/targets/")
    .drop("category", "target")
    .withColumn(
        "uma_status",
        F.when(F.trim(F.col("uma_status")) == "", F.lit("proposed"))
        .otherwise(F.col("uma_status"))
    )       
)
column_types = [("condition_id", "string"),
                ("side", "string"),
                ("outcome", "string"),
                ("price", "double"),
                ("size", "double"),
                ("timestamp", "long"),
                ("asset", "string")]

trades_schema = SparkUtils.generate_schema(column_types)

trades = su._spark.read.schema(trades_schema).parquet("/opt/spark/work-dir/data/labels/trades/")

trades.show()

26/04/18 05:03:16 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


+--------------------+----+-------+-------------------+------------------+----------+--------------------+
|        condition_id|side|outcome|              price|              size| timestamp|               asset|
+--------------------+----+-------+-------------------+------------------+----------+--------------------+
|0x0000dbb9f89318f...| BUY|     No| 0.7900000214576721| 37.97468185424805|1773535485|54757513913264170...|
|0x0000dbb9f89318f...| BUY|    Yes| 0.8500000238418579|               5.0|1773554079|89451665169042906...|
|0x0000dbb9f89318f...| BUY|    Yes| 0.8499999046325684|1.1764659881591797|1773569429|89451665169042906...|
|0x0000dbb9f89318f...| BUY|    Yes| 0.8199999928474426|1.2195110321044922|1773594139|89451665169042906...|
|0x0000dbb9f89318f...| BUY|     No| 0.5401785373687744| 20.36363410949707|1773636999|54757513913264170...|
|0x0000dbb9f89318f...| BUY|     No| 0.5462962985038757|18.305082321166992|1773652055|54757513913264170...|
|0x0000dbb9f89318f...| BUY|     No|  

In [58]:
su._spark.stop()